# 04_features.ipynb (v3)
**DyPriZa – Feature Engineering** (coherente con 00–03 y 02 v2)

__Entrada__: `data/DyPriZa_MVP.csv` (generado por `02_clean_market_data.ipynb` v2)
__Salida__:  `data/DyPriZa_features.csv`

Incluye:
- Variables temporales y estacionales
- Codificación cíclica (mes, día_semana)
- Lags y medias móviles por propiedad/ciudad
- One-Hot Encoding de categóricas (temporada, ciudad, día_semana)
- Salvaguardas y detección flexible de columnas

In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path 

INPUT_PATH ='data/DyPriZa_MVP.csv'
OUTPUT_PATH ='data/DyPriZa_features.csv'
Path ('data').mkdir (exist_ok =True )

data =pd .read_csv (INPUT_PATH )
print ('✅ MVP cargado:',data .shape )
data .columns =[c .strip ().lower ()for c in data .columns ]
data .head (3 )


✅ MVP cargado: (500, 11)


,is_canceled,adr,arrival_date,lead_time,stay_nights,guests,hotel,market_segment,country,reserved_room_type,assigned_room_type
0,0,60.76,2023-08-28,35,7,1,Resort,OTA,FR,C,C
1,0,133.04,2023-04-09,80,2,2,City,OTA,UK,B,B
2,0,58.93,2023-10-21,58,11,4,City,OTA,FR,A,A


## 1) Detección de columnas y tipado seguro

In [5]:



import re 
import numpy as np 
import pandas as pd 

cols_originales =list (data .columns )
data .columns =[str (c ).strip ()for c in data .columns ]
lower_map ={c :c .lower ()for c in data .columns }
data_ =data .rename (columns =lower_map )

def _best_date_column (df :pd .DataFrame )->str |None :

    date_candidates =[
    'fecha','fecharegistro','fechacreate','fec','fch','f_in','f_out',
    'date','created_at','checkin','check_in','arrival','ds'
    ]
    for cand in date_candidates :
        for col in df .columns :
            if col ==cand or col .endswith (f"_{cand}")or cand in col :
                try :
                    parsed =pd .to_datetime (df [col ],errors ='coerce',dayfirst =True )
                    if parsed .notna ().mean ()>0.5 :
                        return col 
                except Exception :
                    pass 


    best_col ,best_rate =None ,0.0 
    for col in df .columns :

        if df [col ].dtype .kind in "biufc":
            continue 
        try :
            parsed =pd .to_datetime (df [col ],errors ='coerce',dayfirst =True )
            rate =parsed .notna ().mean ()

            if rate >best_rate and parsed .nunique (dropna =True )>=max (5 ,int (0.01 *len (parsed ))):
                best_col ,best_rate =col ,rate 
        except Exception :
            continue 
    return best_col 

def _best_price_column (df :pd .DataFrame )->str |None :

    price_candidates =[
    'precio','precio_noche','precio_base','tarifa','tarifa_base','rate','nightly_rate',
    'price','base_price','adr','avg_daily_rate'
    ]
    for cand in price_candidates :
        for col in df .columns :
            if col ==cand or col .endswith (f"_{cand}")or cand in col :
                try :
                    s =pd .to_numeric (df [col ],errors ='coerce')
                    if s .notna ().mean ()>0.5 and s .max ()>0 :
                        return col 
                except Exception :
                    pass 

    numeric_cols =df .select_dtypes (include =[np .number ]).columns .tolist ()
    best_col ,best_score =None ,-1 
    for col in numeric_cols :
        s =pd .to_numeric (df [col ],errors ='coerce')
        score =(s .notna ().mean ()*1000 )+s .var ()
        if s .max ()>0 and s .min ()>=0 and score >best_score :
            best_col ,best_score =col ,score 
    return best_col 


c_fecha =_best_date_column (data_ )
c_precio =_best_price_column (data_ )


def pick_any (df ,names ):
    for n in names :
        if n in df .columns :
            return n 
    for col in df .columns :
        if any (n in col for n in names ):
            return col 
    return None 

c_ocup =pick_any (data_ ,['ocupacion','occupancy','occ_rate','occ'])
c_ciudad =pick_any (data_ ,['ciudad','city','market'])
c_prop =pick_any (data_ ,['propiedad','listing_id','property_id','id_propiedad','idpropiedad','id_listing'])


if c_fecha is None :
    raise AssertionError (
    "No se detectó columna de fecha. "
    f"Columnas disponibles: {cols_originales}. "
    "Sugerencia: renombra tu columna de fechas a algo como 'fecha', 'date', 'checkin' o 'created_at'."
    )
if c_precio is None :
    raise AssertionError (
    "No se detectó columna de precio. "
    f"Columnas disponibles: {cols_originales}. "
    "Sugerencia: renombra tu columna a algo como 'precio', 'price', 'tarifa' o 'adr'."
    )


data [c_fecha ]=pd .to_datetime (data [c_fecha ],errors ='coerce',dayfirst =True )
data [c_precio ]=pd .to_numeric (data [c_precio ],errors ='coerce')
if c_ocup :
    data [c_ocup ]=pd .to_numeric (data [c_ocup ],errors ='coerce')

print ("✅ Columnas detectadas:")
print ({
'fecha':c_fecha ,
'precio':c_precio ,
'ocupacion':c_ocup ,
'ciudad':c_ciudad ,
'propiedad':c_prop 
})


✅ Columnas detectadas:
{'fecha': 'arrival_date', 'precio': 'adr', 'ocupacion': None, 'ciudad': 'market_segment', 'propiedad': None}


C:\Users\nuria\AppData\Local\Temp\ipykernel_7684\506048268.py:23: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
C:\Users\nuria\AppData\Local\Temp\ipykernel_7684\506048268.py:103: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  data[c_fecha]  = pd.to_datetime(data[c_fecha], errors='coerce', dayfirst=True)


In [7]:
print (list (data .columns ))


['is_canceled', 'adr', 'arrival_date', 'lead_time', 'stay_nights', 'guests', 'hotel', 'market_segment', 'country', 'reserved_room_type', 'assigned_room_type']


## 2) Variables temporales, estacionales y codificación cíclica

In [9]:
df =data .copy ()
df ['anio']=df [c_fecha ].dt .year 
df ['mes']=df [c_fecha ].dt .month 
df ['dia']=df [c_fecha ].dt .day 
df ['dow']=df [c_fecha ].dt .weekday 
df ['dia_semana']=df [c_fecha ].dt .day_name ()
df ['fin_de_semana']=df ['dow'].isin ([5 ,6 ]).astype (int )

def temporada (m ):
    if m in [6 ,7 ,8 ]:
        return 'alta'
    elif m in [3 ,4 ,5 ,9 ,10 ]:
        return 'media'
    return 'baja'
df ['temporada']=df ['mes'].apply (temporada )


df ['mes_sin']=np .sin (2 *np .pi *df ['mes']/12 )
df ['mes_cos']=np .cos (2 *np .pi *df ['mes']/12 )
df ['dow_sin']=np .sin (2 *np .pi *df ['dow']/7 )
df ['dow_cos']=np .cos (2 *np .pi *df ['dow']/7 )
print ('✅ Temporales, estacionales y cíclicas creadas')


✅ Temporales, estacionales y cíclicas creadas


## 3) Lags y medias móviles por grupo (propiedad o ciudad)

In [11]:
group_col =c_prop or c_ciudad 
if group_col is None :
    group_col =c_fecha 


df =df .sort_values (by =[group_col ,c_fecha ])


df ['precio_lag_1']=df .groupby (group_col )[c_precio ].shift (1 )
df ['precio_lag_7']=df .groupby (group_col )[c_precio ].shift (7 )
df ['precio_7d_media']=df .groupby (group_col )[c_precio ].transform (lambda x :x .rolling (7 ,min_periods =1 ).mean ())
df ['precio_30d_media']=df .groupby (group_col )[c_precio ].transform (lambda x :x .rolling (30 ,min_periods =1 ).mean ())

if c_ocup :
    df ['ocup_lag_7']=df .groupby (group_col )[c_ocup ].shift (7 )
    df ['ocup_7d_media']=df .groupby (group_col )[c_ocup ].transform (lambda x :x .rolling (7 ,min_periods =1 ).mean ())
    df ['ocup_30d_media']=df .groupby (group_col )[c_ocup ].transform (lambda x :x .rolling (30 ,min_periods =1 ).mean ())


df ['precio_vs_7d']=df [c_precio ]/(df ['precio_7d_media']+1e-9 )
df ['precio_vs_30d']=df [c_precio ]/(df ['precio_30d_media']+1e-9 )
print ('✅ Lags y rolling creados')


✅ Lags y rolling creados


## 4) Codificación categórica (One-Hot) y limpieza final

In [13]:
cat_cols =['temporada','dia_semana']
if c_ciudad :
    cat_cols .append (c_ciudad )
df =pd .get_dummies (df ,columns =cat_cols ,drop_first =True )


before =df .shape [0 ]
df =df .drop_duplicates ()
df =df .dropna (subset =[c_precio ])
print ('🧹 Filas eliminadas (dup/NaN precio):',before -df .shape [0 ])

df .to_csv (OUTPUT_PATH ,index =False )
print ('💾 Guardado:',OUTPUT_PATH )
print ('Shape final:',df .shape )
df .head (3 )


🧹 Filas eliminadas (dup/NaN precio): 0
💾 Guardado: data/DyPriZa_features.csv
Shape final: (500, 35)


,is_canceled,adr,arrival_date,lead_time,stay_nights,guests,hotel,country,reserved_room_type,assigned_room_type,...,temporada_baja,temporada_media,dia_semana_Monday,dia_semana_Saturday,dia_semana_Sunday,dia_semana_Thursday,dia_semana_Tuesday,dia_semana_Wednesday,market_segment_Direct,market_segment_OTA
43,1,134.33,2023-01-01,119,10,1,City,DE,B,C,...,True,False,False,False,True,False,False,False,False,False
195,1,94.61,2023-01-05,3,11,4,Resort,UK,C,B,...,True,False,False,False,False,True,False,False,False,False
171,0,57.06,2023-01-06,94,1,2,Resort,ES,C,A,...,True,False,False,False,False,False,False,False,False,False


### Notas
- Este archivo **no modifica el objetivo** (`precio`), solo crea explicativas adicionales.
- Si el dataset es muy corto, algunas columnas `lag_7` pueden quedar con `NaN` al inicio del grupo.
- Puedes añadir señales externas (festivos, eventos, clima) en futuras versiones.